<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->
# SpiceManager and Celestial Bodies
---
*Scarabaeus | Last revised 2026*

## What this notebook covers
Core infrastructure classes for SPICE kernel interaction and planetary body representation.

### Topics
| # | Topic |
|---|-------|
| 1 | SpiceManager — kernel management |
| 2 | SpiceManager — time-system conversions |
| 3 | SpiceManager — reference-frame transformations |
| 4 | SpiceManager — body state queries |
| 5 | CelestialBody — creation and physical constants |
| 6 | CelestialBody — ephemeris queries |

## How to run
Run from the **project root** directory (`scarabaeus/`).

## 0. Imports and Setup

In [1]:
import os
import scarabaeus as scb
import supplementary as supp
import numpy as np
import matplotlib.pyplot as plt

if os.path.basename(os.getcwd()) == 'tutorials':
    os.chdir('..')
print(f"Working directory : {os.getcwd()}")

data = supp.load_data()

# ── units and frames ──────────────────────────────────────────────
km, AU, kg, sec, day, mu = scb.Units.get_units(['km', 'AU', 'kg', 'sec', 'day', 'mu'])
J2000, ECLIPJ2000, IAU_EARTH, IAU_JUPITER = (
    scb.Frame('J2000'), scb.Frame('ECLIPJ2000'),
    scb.Frame('IAU_EARTH'), scb.Frame('IAU_JUPITER'),
)

# ── kernel pool ───────────────────────────────────────────────────
scb.SpiceManager.clear_kernels()
scb.SpiceManager.load_kernel_from_mkfile(data.mk.path)
scb.SpiceManager.print_kernels()

# ── global plot style ─────────────────────────────────────────────
plt.rcParams.update({
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'legend.fontsize': 8,
    'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.35, 'grid.linestyle': '--',
})

Working directory : /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/tutorials/supplementary
SCB tutorial data up to date.
                                 Kernels Loaded:
Source:   /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/tutorials/supplementary/data/kernels/locked/locked_generic.tm   (META)
Source:   data/kernels/locked/ck/cas00084.tsc   (TEXT)
Source:   data/kernels/locked/lsk/naif0012.tls   (TEXT)
Source:   data/kernels/locked/spk/de432s.bsp   (SPK)
Source:   data/kernels/locked/pck/pck00010.tpc   (TEXT)
Source:   data/kernels/locked/pck/gm_de431.tpc   (TEXT)
Source:   data/kernels/locked/spk/earthstns_fx_201023.bsp   (SPK)
Source:   data/kernels/locked/spk/earth_200101_990628_predict.bpc   (PCK)
Source:   data/kernels/locked/spk/earth_topo_201023.tf   (TEXT)


---
## 1. SpiceManager

`SpiceManager` is the Scarabaeus gateway to the SPICE toolkit.
Every time-system conversion, reference-frame rotation matrix, and body-state vector that
Scarabaeus computes internally passes through this class.

Kernel management was already handled in the setup cell above:
`clear_kernels()` resets the pool, `load_kernel_from_mkfile` furnishes the meta-kernel,
and `print_kernels()` lists what is loaded.
The next three sections cover the utility methods you will call directly.

### Time Conversions

SPICE uses **Ephemeris Time (ET)** internally — seconds past J2000.0 TDB.
`SpiceManager` provides round-trip utilities between ET and every common astronomical time format.

| Method | Direction |
|---|---|
| `et2jd(et)` | ET → Julian Date |
| `et2cal(et)` | ET → calendar string |
| `et2utc(et)` | ET → UTC string |
| `jd2et(jd)` | Julian Date → ET |
| `cal2et(cal)` | calendar string → ET |
| `utc2et(utc)` | UTC string → ET |
| `str2et(s)` | generic SPICE string → ET |
| `et2sclk` / `sclk2et` | ET ↔ spacecraft clock ticks (SCLK kernel required) |

In [2]:
# reference epoch: 2004-JUN-11 ~19:32 TDB
et  = 140254384.184625
jd  = 2453168.3138889
utc = "2004-06-10 19:32:00"
cal = "2004 JUN 11 19:31:59.9999997914"

print("── From ET ──────────────────────────────────────────")
print(f"  ET → JD : {scb.SpiceManager.et2jd(et)}")
print(f"  ET  → CAL : {scb.SpiceManager.et2cal(et)}")
print(f"  ET  → UTC : {scb.SpiceManager.et2utc(et)}")

print("\n── To ET ────────────────────────────────────────────")
print(f"  UTC → ET  : {scb.SpiceManager.utc2et(utc)}")
print(f"  CAL → ET  : {scb.SpiceManager.cal2et(cal)}")
print(f"  JD  → ET  : {scb.SpiceManager.jd2et(jd)}")
print(f"  str → ET  : {scb.SpiceManager.str2et('2004 JUN 11 19:31:59.999 TDB')}")

── From ET ──────────────────────────────────────────
  ET → JD : JD 2453168.3138889
  ET  → CAL : 2004 JUN 11 19:31:59.9999997913837
  ET  → UTC : 2004-06-11T19:31:59.999999

── To ET ────────────────────────────────────────────
  UTC → ET  : 140167984.1846511
  CAL → ET  : 140254384.184625
  JD  → ET  : 140254384.18558368
  str → ET  : 140254319.999


### Reference-Frame Transformations

`get_xfrm(from_frame, to_frame, et)` returns the 3×3 rotation matrix that maps vectors
expressed in `from_frame` into `to_frame` at epoch `et`.
Frame names follow the SPICE convention (`"J2000"`, `"ECLIPJ2000"`, `"IAU_EARTH"`, …).

In [3]:
print("ECLIPJ2000 → J2000 :")
print(scb.SpiceManager.get_xfrm(ECLIPJ2000.name, J2000.name, et))

print("\nECLIPJ2000 → IAU_EARTH :")
print(scb.SpiceManager.get_xfrm(ECLIPJ2000.name, IAU_EARTH.name, et))

ECLIPJ2000 → J2000 :
[[ 1.          0.          0.        ]
 [ 0.          0.91748206 -0.39777716]
 [ 0.          0.39777716  0.91748206]]

ECLIPJ2000 → IAU_EARTH :
[[-9.72925256e-01 -2.11880898e-01  9.23197279e-02]
 [ 2.31119580e-01 -8.92681357e-01  3.86915668e-01]
 [ 4.32060908e-04  3.97776922e-01  9.17482062e-01]]


### Body State Queries

For any body in the kernel pool, `SpiceManager` returns its position, velocity, or full
state relative to an observer in a chosen reference frame.

| Method | Returns |
|---|---|
| `get_pos(target, et, frame, observer)` | 3-vector position (km) |
| `get_vel(target, et, frame, observer)` | 3-vector velocity (km/s) |
| `get_state(target, et, frame, observer)` | 6-vector \[pos; vel\] |
| `get_lighttime(target, et, frame, observer)` | one-way light-time (s) |

In [4]:
# ── Earth w.r.t. Sun at the reference epoch ───────────────────────
print("── Earth w.r.t. Sun  (J2000) ────────────────────────────")
print(f"  pos    (km) : {scb.SpiceManager.get_pos('Earth', et, J2000.name, 'Sun')}")
print(f"  vel  (km/s) : {scb.SpiceManager.get_vel('Earth', et, J2000.name, 'Sun')}")
print(f"  state       : {scb.SpiceManager.get_state('Earth', et, J2000.name, 'Sun')}")
print(f"  lighttime(s): {scb.SpiceManager.get_lighttime('Earth', et, J2000.name, 'Sun')}")

── Earth w.r.t. Sun  (J2000) ────────────────────────────
  pos    (km) : [-2.34128282e+07 -1.37716948e+08 -5.97056599e+07] [km ... km]
  vel  (km/s) : [28.94979027 -4.32365437 -1.87551229] [km/sec ... km/sec]
  state       : [-2.34128282e+07 -1.37716948e+08 -5.97056599e+07  2.89497903e+01
 -4.32365437e+00 -1.87551229e+00] [non-homogeneous]
  lighttime(s): 506.7417668207776 sec


---
## 2. CelestialBody

A `CelestialBody` bundles the physical constants of a planet or natural satellite
(mass, mean radius, gravitational parameter) with a SPICE identifier for ephemeris queries.
There are two construction paths.

### First Method: SCB Constants
The quickest approach — pulls all values from Scarabaeus' built-in `Constants` table:

In [5]:
jupiter_from_constants = scb.CelestialBody.from_constants('JUPITER')
print("\n── Jupiter from constants ──────────────────────────────")
print(f"{'barycenter_id':<25}: {jupiter_from_constants.barycenter_id}")
print(f"{'base_frame':<25}: {jupiter_from_constants.base_frame}")
print(f"{'grav_param':<25}: {jupiter_from_constants.grav_param} mu")
print(f"{'mass_profile':<25}: {jupiter_from_constants.mass_profile} kg")
print(f"{'mean_radius':<25}: {jupiter_from_constants.mean_radius} km")
print(f"{'name':<25}: {jupiter_from_constants.name}")
print(f"{'spice_id':<25}: {jupiter_from_constants.spice_id}")


── Jupiter from constants ──────────────────────────────
barycenter_id            : 5
base_frame               : IAU_JUPITER
grav_param               : 126686534.92180079 mu mu
mass_profile             : 1.898124671078627e+27 kg kg
mean_radius              : 69911.0 km km
name                     : Jupiter
spice_id                 : 599


### Second Method: Explicit Constructor

Supply your own values when the body is not in Scarabaeus' catalogue or you need
non-default constants:

In [6]:
jupiter_custom = scb.CelestialBody(name        = 'JUPITER_CUSTOM',
                                   mass        = scb.ArrayWUnits(1.9e27, kg),
                                   mean_radius = scb.ArrayWUnits(71000, km),
                                   grav_param  = scb.ArrayWUnits(1.3e8, mu),
                                   base_frame  = IAU_JUPITER,
                                   spice_id    = 599000)

`disp_properties()` prints the full set of physical constants — useful for a side-by-side comparison.

In [7]:
jupiter_from_constants.disp_properties()
jupiter_custom.disp_properties()

                           Jupiter                            
mass                     : <bound method Body.mass of Jupiter>
mean_radius              : 69911.0 km
gravitational parameter  : 126686534.92180079 mu
reference frame          : IAU_JUPITER
SPICE ID                 : 599
attached ground stations : []
                            JUPITER_CUSTOM                           
mass                     : <bound method Body.mass of JUPITER_CUSTOM>
mean_radius              : 71000.0 km
gravitational parameter  : 130000000.0 mu
reference frame          : IAU_JUPITER (599 - JUPITER)
SPICE ID                 : 599000
attached ground stations : []


### Querying Ephemerides

If the body exists in the kernel pool, `get_state` returns its position and velocity at any epoch.
Below we query all eight planets' heliocentric orbits over 1 month and plot them.
Note `cal2et` (§1) converting the calendar-string epoch boundaries.

In [8]:
# ── 10-year time interval ─────────────────────────────────────────
t0_ep = scb.EpochArray(scb.SpiceManager.cal2et('2001 JAN 01 00:00:00.000'), 'TDB')
tf_ep = scb.EpochArray(scb.SpiceManager.cal2et('2001 JAN 30 00:00:00.000'), 'TDB')
epochs = scb.EpochArray.interval(t0_ep, tf_ep, scb.ArrayWUnits(1, day))

# ── query heliocentric state for each planet ──────────────────────
planet_names = ['MERCURY', 'VENUS', 'EARTH']
planet_traj  = {}
for name in planet_names:
    cb     = scb.CelestialBody.from_constants(name)
    states = []
    for epoch in epochs:
        state = cb.get_state(epoch_0=epoch, reference_frame='J2000',
                             origin='SUN').quantity
        states.append(state[0:2].convert_to(AU).values)
    planet_traj[name] = np.array(states)
    print(f"  {name:<10s} — {len(states)} epochs queried")

# ── plot ──────────────────────────────────────────────────────────
COLORS = [plt.cm.tab10(i / 10) for i in range(10)]

fig, ax = plt.subplots(figsize=(7, 7))
fig.suptitle('CelestialBody — Heliocentric Orbits JAN 01 2001 to JAN 30 2001 (TDB)',
             fontweight='bold', fontsize=11)

for i, name in enumerate(planet_names):
    xy = planet_traj[name]
    ax.plot(xy[:, 0], xy[:, 1], color=COLORS[i], alpha=0.6, lw=1.2)
    ax.scatter(xy[-1, 0], xy[-1, 1], color=COLORS[i], marker='.', s=40, label=name)

# ── inset: inner solar system ─────────────────────────────────────
det = ax.inset_axes([0.0, 0.03, 0.5, 0.5],
                    xlim=(-1.75, 1.75), ylim=(-1.75, 1.75),
                    xticklabels=[], yticklabels=[])
for i, name in enumerate(planet_names[:4]):
    xy = planet_traj[name]
    det.plot(xy[:, 0], xy[:, 1], color=COLORS[i], alpha=0.6, lw=1.2)
    det.scatter(xy[-1, 0], xy[-1, 1], color=COLORS[i], marker='o', s=20)
det.plot(0, 0, '*', color='goldenrod', ms=8)
det.set_aspect('equal')
ax.indicate_inset_zoom(det, edgecolor='k')

ax.legend(ncol=2, fontsize=8)
ax.set_aspect('equal')
ax.set_xlabel('X [AU]  (J2000)')
ax.set_ylabel('Y [AU]  (J2000)')
plt.tight_layout()
plt.show()

TypeError: ArrayWUnits._are_shapes_compatible() missing 1 required positional argument: 'other'